# Signal Hunters - Team Cyclops (Hack4Hack 2026 Round 2)

Runs the code from the team repository end to end on Kaggle.

**Before running:** add the Round 2 data as an input dataset (any name; the notebook finds
`train_cadences.npy`), set **Accelerator: GPU T4** and turn **Internet on** (the pretrained
backbones download public ImageNet weights through timm). A full run takes about 8 hours.

Steps: folds -> features (train, PKS-degraded train with gain jitter, test) -> LightGBM ->
three small CNNs and three ImageNet-pretrained EfficientNets on the global-time image ->
second-level stack -> per-class bias -> submissions. The final file is
`submissions/18_stack7_bias.csv`. All outputs land in `/kaggle/working/signal-hunters/submissions`
and `/kaggle/working/signal-hunters/oof`.

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)
assert hits, 'Add the Round 2 data as an input dataset first'
os.environ['SIG_DATA'] = os.path.dirname(hits[0])
os.environ['SIG_FEATURES'] = '/kaggle/working/features'
for d in ['src/sig', 'scripts', 'oof', 'submissions']:
    os.makedirs(f'/kaggle/working/signal-hunters/{d}', exist_ok=True)
os.chdir('/kaggle/working/signal-hunters')
print(os.environ['SIG_DATA'], sorted(os.listdir(os.environ['SIG_DATA'])))
!nvidia-smi -L; nproc

## 0. Repository code (written verbatim)

In [ ]:
%%writefile src/sig/data.py
"""Paths, labels and loaders shared by every script."""

import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
DATA = Path(os.environ.get("SIG_DATA", ROOT / "data"))              # read-only inputs (Kaggle: /kaggle/input/...)
FEATURES = Path(os.environ.get("SIG_FEATURES", DATA / "features"))  # derived feature tables
CPUS = os.cpu_count() or 4
SEED = 42
LABELS = sorted([
    "eti_candidate", "frb", "noise", "pulsar", "radar_chirp",
    "rfi_intermittent", "rfi_persistent", "satellite_pass", "scintillating_star",
])
PRIORITY = [
    "eti_candidate", "frb", "pulsar", "scintillating_star", "satellite_pass",
    "radar_chirp", "rfi_intermittent", "rfi_persistent", "noise",
]
N_PANELS, N_ROWS, N_CHAN = 6, 16, 128
ON = np.array([True, False, True, False, True, False])


@dataclass
class Split:
    name: str
    x: np.ndarray        # (n, 6, 16, 128) uint8, 0 = dropped row
    meta: pd.DataFrame

    @property
    def gaps(self) -> np.ndarray:
        return self.meta["slew_gap_rows"].to_numpy()

    @property
    def y(self) -> np.ndarray:
        return self.meta["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()


def load_split(name: str, mmap: bool = True) -> Split:
    x = np.load(DATA / f"{name}_cadences.npy", mmap_mode="r" if mmap else None)
    meta = pd.read_csv(DATA / f"{name}_meta.csv")
    return Split(name, x, meta)


def load_folds() -> np.ndarray:
    return pd.read_csv(ROOT / "folds.csv")["fold"].to_numpy()


def global_rows(gap: int) -> np.ndarray:
    """Global time step of every (panel, row): panel p, row r is at p * (16 + gap) + r."""
    return (np.arange(N_PANELS)[:, None] * (N_ROWS + gap) + np.arange(N_ROWS)[None, :])


In [ ]:
%%writefile src/sig/prep.py
"""Telescope-neutral normalization of a batch of cadences.

Steps, all per cadence (n, 6, 16, 128) in log2-power units (byte / 24):
1. Dropped rows (all zeros) become missing.
2. Subtract each row's median over channels: removes panel gain changes and broadband glitch rows.
3. Subtract a smooth bandpass: the per-channel median over all 96 rows, running-median smoothed
   across channels. Narrow lines survive (a persistent RFI line is too thin to bend the smooth
   curve); ripple, roll-off and tilt do not.
4. The GBT centre-channel spike is replaced by its neighbours on every cadence, so neither
   telescope has it.
5. Divide by the robust noise level of the cadence.
"""

import numpy as np
from scipy.ndimage import median_filter

SCALE = 24.0
SMOOTH = 13
SPIKE = 64


PKS_GAMMA_SHAPE = 7.7     # extra multiplicative noise that widens GBT byte MAD 13.3 to the PKS 18.5
PKS_EXTRA_DROP = 0.0023   # PKS drops 0.38% of rows, GBT 0.15%
PKS_GAIN_JITTER = 0.062   # extra per-panel gain spread in log2 power: OFF-panel level std 0.049 (GBT) vs 0.079 (PKS)


def degrade(x: np.ndarray, rng: np.random.Generator, gain_jitter: bool = False) -> np.ndarray:
    """Make GBT cadences look like PKS noise: multiply linear power by Gamma(k)/k noise, drop rows.

    Multiplying power by an independent gamma variable is what averaging fewer spectra does, so
    the noise gets wider and gains the negative log-domain skew seen in PKS, and signals fade with it.
    gain_jitter adds the larger panel-to-panel gain changes of PKS (off by default, so earlier models
    rebuild unchanged).
    """
    g = rng.gamma(PKS_GAMMA_SHAPE, 1 / PKS_GAMMA_SHAPE, size=x.shape).astype(np.float32)
    v = x.astype(np.float32) + SCALE * np.log2(g)
    if gain_jitter:
        v += SCALE * rng.normal(0, PKS_GAIN_JITTER, size=x.shape[:2] + (1, 1)).astype(np.float32)
    out = np.clip(np.round(v), 1, 255).astype(np.uint8)
    drop = rng.random(x.shape[:3]) < PKS_EXTRA_DROP
    out[(x == 0) | drop[..., None]] = 0
    return out


def normalize(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (z, valid): z float32 (n, 6, 16, 128) with 0 at dropped rows; valid (n, 6, 16) bool."""
    f = x.astype(np.float32) / SCALE
    valid = x.max(axis=3) > 0
    f[..., SPIKE] = 0.5 * (f[..., SPIKE - 1] + f[..., SPIKE + 1])

    f -= np.median(f, axis=3, keepdims=True)
    f[~valid] = 0.0

    n = len(f)
    flat = f.reshape(n, -1, f.shape[-1])
    w = valid.reshape(n, -1)
    prof = np.median(flat, axis=1)
    gappy = np.where(~w.all(axis=1))[0]
    if len(gappy):
        prof[gappy] = np.nanmedian(np.where(w[gappy, :, None], flat[gappy], np.nan), axis=1)
    bp = median_filter(prof, size=(1, SMOOTH), mode="nearest")
    f -= bp[:, None, None, :]
    f[~valid] = 0.0

    sig = 1.4826 * np.median(np.abs(f.reshape(n, -1)), axis=1) + 1e-6
    z = f / sig[:, None, None, None]
    z[~valid] = 0.0
    return z.astype(np.float32), valid


In [ ]:
%%writefile src/sig/features.py
"""Hand-built cadence features from normalized cadences z (n, 6, 16, 128), see `sig.prep.normalize`.

Four searches, then the ON/OFF cadence logic on top of them:
- narrowband drift search per panel: S/N of the best straight line through the panel's mid row
  at every (drift, channel);
- global ETI track: one straight line through all six panels on the global time axis
  (`p * (16 + gap) + r`), scored separately on the ON and OFF panels;
- broadband dispersed sweeps per panel (pulsar, FRB, radar): sum over channels along a sweep
  that arrives later at lower frequencies;
- patch texture (scintillation): box-smoothed power over 4 rows x 8 channels.
"""

import numpy as np
import pandas as pd

from sig.data import N_CHAN, N_ROWS, ON

DRIFTS = np.concatenate([np.arange(-12, -6, 1.0), np.arange(-6, 6.01, 0.25), np.arange(7, 12.1, 1.0)])
PAD = 100
SMOOTH_VAR = 6 / 16                     # noise variance left after the [1, 2, 1] / 4 channel smoothing
TRACK_DRIFTS = np.arange(-1.5, 1.5001, 0.02)
STATIC = 0.06
SWEEP_SLOPES = np.array([2, 3, 4, 6, 8, 11, 15, 20, 26, 34], float)
SWEEP_T0 = np.arange(-34, 16)
GROUP = 4


def smooth_channels(z: np.ndarray) -> np.ndarray:
    p = np.pad(z, [(0, 0)] * (z.ndim - 1) + [(1, 1)], mode="edge")
    return 0.25 * p[..., :-2] + 0.5 * p[..., 1:-1] + 0.25 * p[..., 2:]


def drift_maps(z: np.ndarray, nvalid: np.ndarray) -> np.ndarray:
    """(n, 6, D, 128): S/N of a straight line with drift d (channels per row) crossing channel f at row 7.5."""
    zs = np.pad(smooth_channels(z), ((0, 0), (0, 0), (0, 0), (PAD, PAD)))
    r = np.arange(N_ROWS) - 7.5
    out = np.empty(z.shape[:2] + (len(DRIFTS), N_CHAN), np.float32)
    for i, d in enumerate(DRIFTS):
        sh = np.round(d * r).astype(int)
        acc = np.zeros(z.shape[:2] + (N_CHAN,), np.float32)
        for k in range(N_ROWS):
            acc += zs[:, :, k, PAD + sh[k]:PAD + sh[k] + N_CHAN]
        out[:, :, i] = acc
    return out / np.sqrt(np.maximum(nvalid, 1) * SMOOTH_VAR)[:, :, None, None]


def track_scores(s: np.ndarray, gaps: np.ndarray, chunk: int = 50) -> dict:
    parts = [_track_scores(s[a:a + chunk], gaps[a:a + chunk]) for a in range(0, len(s), chunk)]
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}


def _track_scores(s: np.ndarray, gaps: np.ndarray) -> dict:
    """Score straight global-time tracks through the panel maps s (n, 6, D, 128).

    For a track with drift g crossing channel c at the middle of ON panel 3 (index 2), panel p is
    crossed at channel c + g * (p - 2) * (16 + gap). Each panel's S/N is read at its nearest local drift.
    """
    n = len(s)
    pooled = np.maximum(s, np.maximum(np.roll(s, 1, -1), np.roll(s, -1, -1)))
    di = np.abs(DRIFTS[None, :] - TRACK_DRIFTS[:, None]).argmin(1)                     # (G,)
    sub = pooled[:, :, di]                                                              # (n, 6, G, 128)
    dp = (np.arange(6) - 2)[None, :, None] * (N_ROWS + gaps)[:, None, None]             # (n, 6, 1)
    off = np.round(TRACK_DRIFTS[None, None, :] * dp).astype(np.int32)                       # (n, 6, G)
    pos = np.arange(N_CHAN)[None, None, None, :] + off[..., None]                       # (n, 6, G, 128)
    inside = (pos >= 0) & (pos < N_CHAN)
    vals = np.take_along_axis(sub, np.clip(pos, 0, N_CHAN - 1), axis=3)
    vals = np.where(inside, vals, np.float32(0))                                                 # out of band: no evidence
    on, offp = vals[:, ON], vals[:, ~ON]
    min_on = on.min(1)                                                                  # (n, G, 128)
    max_off = offp.max(1)
    clean = min_on - np.maximum(max_off, 0)
    static = np.abs(TRACK_DRIFTS) < STATIC
    feats = {}
    flat = min_on.reshape(n, -1)
    best = flat.argmax(1)
    g_best, c_best = np.unravel_index(best, min_on.shape[1:])
    feats["trk_min_on"] = flat.max(1)
    feats["trk_drift"] = TRACK_DRIFTS[g_best]
    feats["trk_abs_drift"] = np.abs(TRACK_DRIFTS[g_best])
    feats["trk_off_max_at_best"] = max_off[np.arange(n), g_best, c_best]
    feats["trk_off_mean_at_best"] = offp.mean(1)[np.arange(n), g_best, c_best]
    feats["trk_on_sum_at_best"] = on.sum(1)[np.arange(n), g_best, c_best] / np.sqrt(3)
    feats["trk_clean"] = clean.reshape(n, -1).max(1)
    feats["trk_clean_drifting"] = clean[:, ~static].reshape(n, -1).max(1)
    feats["trk_clean_static"] = clean[:, static].reshape(n, -1).max(1)
    feats["trk_min_on_drifting"] = min_on[:, ~static].reshape(n, -1).max(1)
    feats["trk_min_on_static"] = min_on[:, static].reshape(n, -1).max(1)
    feats["trk_sum_on"] = on.sum(1).reshape(n, -1).max(1) / np.sqrt(3)
    feats["trk_min_off"] = offp.min(1).reshape(n, -1).max(1)
    # Coherent sum over the three ON crossings with an OFF penalty: faint ETI that no single panel shows
    sum_clean = on.sum(1) / np.sqrt(3) - np.maximum(max_off, 0)
    feats["trk_sum_clean_drifting"] = sum_clean[:, ~static].reshape(n, -1).max(1)
    feats["trk_sum_clean_static"] = sum_clean[:, static].reshape(n, -1).max(1)
    sc = sum_clean[:, ~static].reshape(n, -1)
    top = sc.argmax(1)
    g2, c2 = np.unravel_index(top, sum_clean[:, ~static].shape[1:])
    feats["trk_sum_clean_on_min"] = on[:, :, ~static].min(1)[np.arange(n), g2, c2]
    feats["trk_sum_clean_drift"] = np.abs(TRACK_DRIFTS[~static][g2])
    feats.update(_drift_consistency(pooled, pos[:, :, ~static], TRACK_DRIFTS[~static], g2, c2))
    feats["_best_g"] = TRACK_DRIFTS[~static][g2]
    feats["_best_c"] = c2
    return feats


def _drift_consistency(pooled: np.ndarray, pos: np.ndarray, drifts: np.ndarray, g: np.ndarray, c: np.ndarray) -> dict:
    """At the best drifting track, does each ON panel's own line drift like the track does?

    A real ETI signal has one drift rate everywhere; the "lines that do not connect" trap has
    panel lines whose local drift disagrees with the slope joining them.
    """
    n = len(pooled)
    idx = np.arange(n)
    gd = drifts[g]
    d_err, s_ratio = [], []
    for p in np.where(ON)[0]:
        f = np.clip(pos[idx, p, g, c], 0, N_CHAN - 1)
        window = np.stack([pooled[idx, p, :, np.clip(f + k, 0, N_CHAN - 1)] for k in (-2, -1, 0, 1, 2)], -1)
        local = window.max(-1)                                                             # (n, D)
        best = local.argmax(1)
        d_err.append(np.abs(DRIFTS[best] - gd))
        at_g = local[idx, np.abs(DRIFTS[None, :] - gd[:, None]).argmin(1)]
        s_ratio.append(at_g / (local.max(1) + 1e-6))
    d_err, s_ratio = np.stack(d_err, 1), np.stack(s_ratio, 1)
    return {
        "trk_cons_drift_err_mean": d_err.mean(1),
        "trk_cons_drift_err_max": d_err.max(1),
        "trk_cons_ratio_min": s_ratio.min(1),
        "trk_cons_ratio_mean": s_ratio.mean(1),
    }


def line_fit(z: np.ndarray, gaps: np.ndarray, g: np.ndarray, c: np.ndarray, half: int = 4) -> dict:
    """Sub-channel check of the best drifting track in the three ON panels.

    For every row, the power-weighted centroid of the line within +-4 channels of the track gives
    its offset from the track. A real ETI signal sits on the track everywhere, so the offsets stay
    flat inside each panel (same drift) and equal between panels (one connected line). The trap
    "same drift rate but not connected" shows up as different mean offsets per panel; a drift
    mismatch as a slope inside the panels.
    """
    n = len(z)
    idx = np.arange(n)[:, None, None]
    k = np.arange(-half, half + 1)
    r = np.arange(N_ROWS) - 7.5
    offs, wsum = [], []
    for p in np.where(ON)[0]:
        t = (p - 2) * (N_ROWS + gaps)[:, None] + r[None, :]                                # (n, 16)
        e = c[:, None] + g[:, None] * t                                                    # expected channel
        ch = np.round(e)[..., None].astype(int) + k                                        # (n, 16, 9)
        ok = (ch >= 0) & (ch < N_CHAN)
        v = z[idx, p, np.arange(N_ROWS)[None, :, None], np.clip(ch, 0, N_CHAN - 1)]
        w = np.where(ok, np.clip(v - 1.0, 0, None), 0.0)
        ws = w.sum(-1)
        cen = (w * (ch - e[..., None])).sum(-1) / np.maximum(ws, 1e-6)                     # offset from track
        offs.append(cen)
        wsum.append(ws)
    off, ws = np.stack(offs, 1), np.stack(wsum, 1)                                         # (n, 3, 16)
    wn = ws / np.maximum(ws.sum(2, keepdims=True), 1e-6)
    mean_off = (wn * off).sum(2)                                                           # (n, 3)
    rc = r - (wn * r).sum(2, keepdims=True)
    slope = (wn * rc * (off - mean_off[..., None])).sum(2) / np.maximum((wn * rc ** 2).sum(2), 1e-6)
    return {
        "fit_offset_spread": mean_off.max(1) - mean_off.min(1),
        "fit_offset_absmax": np.abs(mean_off).max(1),
        "fit_slope_absmax": np.abs(slope).max(1),
        "fit_slope_mean": np.abs(slope.mean(1)),
        "fit_weight_min": ws.sum(2).min(1),
        "fit_weight_ratio": ws.sum(2).min(1) / np.maximum(ws.sum(2).max(1), 1e-6),
    }


def sweep_maps(z: np.ndarray) -> np.ndarray:
    """(n, 6, shapes * slopes, t0): S/N of a broadband sweep arriving at row t0 at the top channel
    and `slope` rows later at the bottom, either linear or quadratic in channel."""
    n = len(z)
    g = z.reshape(n, 6, N_ROWS, N_CHAN // GROUP, GROUP).sum(-1) / np.sqrt(GROUP)          # (n, 6, 16, 32)
    ng = g.shape[-1]
    u = (ng - 1 - np.arange(ng)) / (ng - 1)                                                # 1 at the lowest channel
    out = []
    for shape in (u, u ** 2):
        for s in SWEEP_SLOPES:
            rows = np.round(SWEEP_T0[:, None] + s * shape[None, :]).astype(int)              # (T0, 32)
            ok = (rows >= 0) & (rows < N_ROWS)
            v = g[:, :, np.clip(rows, 0, N_ROWS - 1), np.arange(ng)[None, :]]              # (n, 6, T0, 32)
            out.append((v * ok).sum(-1) / np.sqrt(np.maximum(ok.sum(-1), 1)))
    return np.stack(out, 2)


def patch_texture(z: np.ndarray) -> np.ndarray:
    """(n, 6, 4, 16): mean S/N of 4-row x 8-channel boxes."""
    n = len(z)
    return z.reshape(n, 6, 4, 4, 16, 8).mean((3, 5)) * np.sqrt(32)


def _panel_block(name: str, v: np.ndarray, cols: dict) -> None:
    """Per-panel values v (n, 6) plus ON/OFF summaries."""
    for p in range(6):
        cols[f"{name}_p{p}"] = v[:, p]
    cols[f"{name}_on_mean"] = v[:, ON].mean(1)
    cols[f"{name}_off_mean"] = v[:, ~ON].mean(1)
    cols[f"{name}_on_min"] = v[:, ON].min(1)
    cols[f"{name}_off_max"] = v[:, ~ON].max(1)
    cols[f"{name}_on_off"] = cols[f"{name}_on_mean"] - cols[f"{name}_off_mean"]


def cadence_features(z: np.ndarray, valid: np.ndarray, gaps: np.ndarray) -> pd.DataFrame:
    n = len(z)
    cols: dict = {}
    nvalid = valid.sum(2)
    cols["n_dropped"] = (~valid).sum((1, 2))

    zv = z.reshape(n, 6, -1)
    _panel_block("std", zv.std(2), cols)
    _panel_block("frac_hi3", (zv > 3).mean(2), cols)
    _panel_block("frac_lo3", (zv < -3).mean(2), cols)
    _panel_block("p999", np.percentile(zv, 99.9, axis=2), cols)

    s = drift_maps(z, nvalid)                                                   # (n, 6, D, 128)
    static = np.abs(DRIFTS) < 0.13
    slow = (np.abs(DRIFTS) >= 0.25) & (np.abs(DRIFTS) <= 2)
    steep = np.abs(DRIFTS) >= 3
    best_d = s.max(3)                                                           # (n, 6, D)
    _panel_block("line_best", best_d.max(2), cols)
    _panel_block("line_static", best_d[:, :, static].max(2), cols)
    _panel_block("line_slow", best_d[:, :, slow].max(2), cols)
    _panel_block("line_steep", best_d[:, :, steep].max(2), cols)
    _panel_block("line_absdrift", np.abs(DRIFTS[best_d.argmax(2)]), cols)
    _panel_block("line_n_steep6", (s[:, :, steep].max(2) > 6).sum(2).astype(float), cols)
    _panel_block("line_n_static6", (s[:, :, static].max(2) > 6).sum(2).astype(float), cols)
    _panel_block("line_n_slow6", (s[:, :, slow].max(2) > 6).sum(2).astype(float), cols)
    cols["n_on_lines6"] = (best_d.max(2)[:, ON] > 6).sum(1)
    cols["n_off_lines6"] = (best_d.max(2)[:, ~ON] > 6).sum(1)
    cols["n_steep_panels8"] = (best_d[:, :, steep].max(2) > 8).sum(1)

    # Same channel in every panel: persistent RFI (allowing one faded panel via the second-lowest)
    st = s[:, :, static].max(2)                                                 # (n, 6, 128)
    st = np.maximum(st, np.maximum(np.roll(st, 1, -1), np.roll(st, -1, -1)))
    srt = np.sort(st, axis=1)
    cols["persist_min6"] = srt[:, 0].max(1)
    cols["persist_2nd6"] = srt[:, 1].max(1)
    cols["persist_on_min"] = st[:, ON].min(1).max(1)
    cols["persist_off_min"] = st[:, ~ON].min(1).max(1)
    cols["persist_n_ch"] = (srt[:, 1] > 5).sum(1)

    trk = track_scores(s, gaps)
    cols.update(line_fit(z, gaps, trk.pop("_best_g"), trk.pop("_best_c")))
    cols.update(trk)
    cols["trk_gap_vs_free"] = cols["trk_min_on"] - cols["line_best_on_min"]

    sw = sweep_maps(z)                                                          # (n, 6, K, T0)
    swb = sw.max(2)                                                             # (n, 6, T0)
    _panel_block("sweep_best", swb.max(2), cols)
    _panel_block("sweep_n4", (swb > 4).sum(2).astype(float), cols)
    k_best = sw.max(3).argmax(2)
    _panel_block("sweep_slope", np.tile(SWEEP_SLOPES, 2)[k_best], cols)
    _panel_block("sweep_curved", (k_best >= len(SWEEP_SLOPES)).astype(float), cols)
    # Repeating pulses: several separated peaks along t0 at the best sweep shape
    kb = np.take_along_axis(sw, k_best[..., None, None], 2)[:, :, 0]            # (n, 6, T0)
    peaks = (kb > 4) & (kb >= np.roll(kb, 1, -1)) & (kb >= np.roll(kb, -1, -1))
    _panel_block("sweep_npeaks", peaks.sum(2).astype(float), cols)

    tx = patch_texture(z).reshape(n, 6, -1)
    _panel_block("patch_std", tx.std(2), cols)
    _panel_block("patch_max", tx.max(2), cols)
    _panel_block("patch_n3", (tx > 3).sum(2).astype(float), cols)
    cols.update(texture_features(z, valid))
    return pd.DataFrame(cols)


def panel_levels(x: np.ndarray) -> pd.DataFrame:
    """Overall brightness of each panel from the raw bytes (n, 6, 16, 128), in log2 power.

    Row-median normalization removes it, but a scintillating star (and a pulsar) lifts the whole
    ON panel. Levels are relative to the cadence mean; ON-OFF contrasts are also given in units
    of the spread between the three OFF panels, which absorbs the telescope's gain jitter.
    """
    f = x.astype(np.float32) / 24.0
    valid = x.max(axis=3) > 0
    rows = np.where(valid, np.median(f, axis=3), np.nan)                                  # (n, 6, 16)
    lvl = np.nanmedian(rows, axis=2)
    lvl = np.nan_to_num(lvl - np.nanmean(lvl, axis=1, keepdims=True))
    on, off = lvl[:, ON], lvl[:, ~ON]
    spread = off.std(1) + 0.02
    row_std = np.nanstd(rows - lvl[..., None], axis=2)
    cols: dict = {}
    _panel_block("lvl", lvl, cols)
    cols["lvl_on_min_off_max"] = on.min(1) - off.max(1)
    cols["lvl_on_off_z"] = (on.mean(1) - off.mean(1)) / spread
    cols["lvl_on_min_off_max_z"] = cols["lvl_on_min_off_max"] / spread
    cols["lvl_off_spread"] = spread
    _panel_block("lvl_rowstd", np.nan_to_num(row_std), cols)
    return pd.DataFrame(cols)


def _corr(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pearson correlation over the last two axes, per (cadence, panel)."""
    a = a - a.mean((-2, -1), keepdims=True)
    b = b - b.mean((-2, -1), keepdims=True)
    return (a * b).sum((-2, -1)) / np.sqrt((a * a).sum((-2, -1)) * (b * b).sum((-2, -1)) + 1e-9)


def texture_features(z: np.ndarray, valid: np.ndarray) -> dict:
    """Scintillation: patchy brightening over many channels that twinkles in time, ON panels only.

    Narrow lines are clipped first so a bright RFI line does not look like texture.
    """
    n = len(z)
    c = np.clip(z, -4, 4)
    cols: dict = {}
    nv = np.maximum(valid.sum(2), 1)
    row_mean = c.mean(3)                                                   # (n, 6, 16): broad excess per row
    _panel_block("tex_mean", c.sum((2, 3)) / (nv * N_CHAN), cols)
    _panel_block("tex_rowmean_std", row_mean.std(2), cols)
    _panel_block("tex_rowmean_max", row_mean.max(2), cols)
    _panel_block("tex_ac_time", _corr(c[:, :, 1:], c[:, :, :-1]), cols)
    _panel_block("tex_ac_chan2", _corr(c[..., 2:], c[..., :-2]), cols)
    _panel_block("tex_ac_chan6", _corr(c[..., 6:], c[..., :-6]), cols)
    _panel_block("tex_skew", ((c - c.mean((2, 3), keepdims=True)) ** 3).mean((2, 3)), cols)
    for rows, chans in ((2, 32), (8, 16), (16, 8)):
        box = c.reshape(n, 6, N_ROWS // rows, rows, N_CHAN // chans, chans).mean((3, 5)) * np.sqrt(rows * chans)
        box = box.reshape(n, 6, -1)
        _panel_block(f"tex_box{rows}x{chans}_std", box.std(2), cols)
        _panel_block(f"tex_box{rows}x{chans}_max", box.max(2), cols)
    return cols


In [ ]:
%%writefile scripts/make_folds.py
"""Write folds.csv: stratified 5-fold split of train, grouped by target star, seed 42."""

import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import ROOT, SEED, load_split  # noqa: E402


def main() -> None:
    tr = load_split("train")
    folds = pd.Series(-1, index=tr.meta.index)
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(skf.split(tr.meta, tr.meta["label"], tr.meta["target"])):
        folds.iloc[va] = k
    out = pd.DataFrame({"id": tr.meta["id"], "fold": folds})
    out.to_csv(ROOT / "folds.csv", index=False)
    print(pd.crosstab(tr.meta["label"], out["fold"]))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/build_features.py
"""Compute hand-built features for train, PKS-degraded train (seed 42) and test.

Writes data/features/{train,train_deg,test}.parquet, one row per cadence in file order.
    python scripts/build_features.py
"""

import argparse
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, SEED, load_split  # noqa: E402
from sig.features import cadence_features, panel_levels  # noqa: E402
from sig.prep import degrade, normalize  # noqa: E402

CHUNK = 250
OUT = FEATURES


def _work(args: tuple) -> pd.DataFrame:
    name, start, stop, deg, jitter = args
    sp = load_split(name)
    x = np.asarray(sp.x[start:stop])
    if deg:
        x = degrade(x, np.random.default_rng([SEED, start]), gain_jitter=jitter)
    z, valid = normalize(x)
    return pd.concat([cadence_features(z, valid, sp.gaps[start:stop]), panel_levels(x)], axis=1)


def build(name: str, deg: bool, workers: int, limit: int | None, jitter: bool) -> pd.DataFrame:
    n = len(load_split(name).meta) if limit is None else limit
    jobs = [(name, a, min(a + CHUNK, n), deg, jitter) for a in range(0, n, CHUNK)]
    with ProcessPoolExecutor(workers) as ex:
        parts = list(ex.map(_work, jobs))
    return pd.concat(parts, ignore_index=True)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=CPUS)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--gain-jitter", action="store_true", help="add PKS panel gain jitter to the degraded copy")
    args = ap.parse_args()
    OUT.mkdir(parents=True, exist_ok=True)
    for name, deg, tag in [("train", False, "train"), ("train", True, "train_deg"), ("test", False, "test")]:
        t = time.time()
        f = build(name, deg, args.workers, args.limit, args.gain_jitter)
        f.to_parquet(OUT / f"{tag}.parquet")
        print(f"{tag}: {f.shape} in {time.time() - t:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/adversarial.py
"""Adversarial validation: how well can LightGBM tell train features from test features?

AUC 0.5 means the two look the same. Prints the AUC and the features that give the telescope away.
    python scripts/adversarial.py --train train_deg
"""

import argparse
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, SEED  # noqa: E402


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train_deg")
    ap.add_argument("--drop", nargs="*", default=[])
    args = ap.parse_args()
    a = pd.read_parquet(FEATURES / f"{args.train}.parquet").drop(columns=args.drop)
    b = pd.read_parquet(FEATURES / "test.parquet").drop(columns=args.drop)
    x = pd.concat([a, b], ignore_index=True)
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    p = np.zeros(len(y))
    imp = 0
    for fit, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(x, y):
        m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=CPUS, verbose=-1)
        m.fit(x.iloc[fit], y[fit])
        p[va] = m.predict_proba(x.iloc[va])[:, 1]
        imp = imp + m.booster_.feature_importance("gain")
    print(f"adversarial AUC {args.train} vs test: {roc_auc_score(y, p):.3f}")
    print(pd.Series(imp, index=x.columns).sort_values(ascending=False).head(20).round(0).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train_lgbm.py
"""LightGBM on the hand-built features, 5 folds from folds.csv.

Trains on original + PKS-degraded train rows, early-stops on the degraded validation fold (the
closest thing to test we have), and writes:
    oof/<name>_oof.csv       degraded-train OOF probabilities (used for blending and tuning)
    oof/<name>_oof_orig.csv  original-train OOF probabilities
    oof/<name>_test.csv      test probabilities, mean over folds
    python scripts/train_lgbm.py --name lgbm
"""

import argparse
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, LABELS, ROOT, SEED, load_folds, load_split  # noqa: E402

PARAMS = dict(
    objective="multiclass", num_class=len(LABELS), learning_rate=0.05, num_leaves=31,
    min_data_in_leaf=40, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=1.0, n_jobs=CPUS, verbose=-1, seed=SEED,
)


def save(path: Path, ids: pd.Series, p: np.ndarray) -> None:
    pd.concat([ids.reset_index(drop=True), pd.DataFrame(p, columns=LABELS)], axis=1).to_csv(path, index=False)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="lgbm")
    ap.add_argument("--no-orig", action="store_true", help="train on degraded rows only")
    args = ap.parse_args()

    tr, te = load_split("train"), load_split("test")
    f_tr = pd.read_parquet(FEATURES / "train.parquet")
    f_dg = pd.read_parquet(FEATURES / "train_deg.parquet")
    f_te = pd.read_parquet(FEATURES / "test.parquet")
    y, folds = tr.y, load_folds()
    n = len(y)

    oof_dg = np.zeros((n, len(LABELS)))
    oof_tr = np.zeros((n, len(LABELS)))
    test = np.zeros((len(f_te), len(LABELS)))
    for k in range(5):
        fit = folds != k
        xs = [f_dg[fit]] if args.no_orig else [f_tr[fit], f_dg[fit]]
        x_fit = pd.concat(xs, ignore_index=True)
        y_fit = np.concatenate([y[fit]] * len(xs))
        d_fit = lgb.Dataset(x_fit, y_fit, weight=compute_sample_weight("balanced", y_fit))
        d_val = lgb.Dataset(f_dg[~fit], y[~fit], weight=compute_sample_weight("balanced", y[~fit]))
        m = lgb.train(PARAMS, d_fit, 3000, valid_sets=[d_val],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_dg[~fit] = m.predict(f_dg[~fit], num_iteration=m.best_iteration)
        oof_tr[~fit] = m.predict(f_tr[~fit], num_iteration=m.best_iteration)
        test += m.predict(f_te, num_iteration=m.best_iteration) / 5
        print(f"fold {k}: iters {m.best_iteration}  degraded F1 "
              f"{f1_score(y[~fit], oof_dg[~fit].argmax(1), average='macro'):.4f}", flush=True)

    print(f"OOF macro-F1  original {f1_score(y, oof_tr.argmax(1), average='macro'):.4f}  "
          f"degraded {f1_score(y, oof_dg.argmax(1), average='macro'):.4f}")
    print(classification_report(y, oof_dg.argmax(1), target_names=LABELS, digits=3))
    cm = pd.DataFrame(confusion_matrix(y, oof_dg.argmax(1)), index=LABELS, columns=[c[:6] for c in LABELS])
    print("confusion (rows true, cols predicted):\n" + cm.to_string())
    print("test predicted mix:", pd.Series(np.array(LABELS)[test.argmax(1)]).value_counts().to_dict())

    imp = pd.Series(m.feature_importance("gain"), index=f_tr.columns).sort_values(ascending=False)
    print("top features:", imp.head(25).round(0).to_dict())
    (ROOT / "oof").mkdir(exist_ok=True)
    save(ROOT / f"oof/{args.name}_oof.csv", tr.meta["id"], oof_dg)
    save(ROOT / f"oof/{args.name}_oof_orig.csv", tr.meta["id"], oof_tr)
    save(ROOT / f"oof/{args.name}_test.csv", te.meta["id"], test)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train_cnn.py
"""2D CNN on the global-time image of each cadence (CPU).

Each cadence becomes a 2 x 128 x 128 image: panel p, row r is placed at global row
p * (16 + gap) + r, so a real ETI signal is one straight line through the three ON panels.
Channel 0 is the normalized power, channel 1 is +1 on ON rows, -1 on OFF rows, 0 in slew gaps
and dropped rows.

Training augmentation: fresh PKS-style degradation (prob 0.75) and a frequency flip (prob 0.5;
it flips the drift sign, which every class allows). Validation uses the fixed degraded copy
made by the same seeds as scripts/build_features.py. Test uses flip TTA.
    python scripts/train_cnn.py --folds 0 1 2 3 4 --name cnn

--arch swaps the small Net for an ImageNet-pretrained timm backbone (mixed precision, needs
internet on the first run to fetch the public weights):
    python scripts/train_cnn.py --arch efficientnet_b0 --levels --lr 1e-3 --epochs 12 --name eff_lvl
"""

import argparse
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, LABELS, N_CHAN, N_ROWS, ON, ROOT, SEED, load_folds, load_split  # noqa: E402
from sig.features import panel_levels  # noqa: E402
from sig.prep import degrade, normalize  # noqa: E402

H = 128
CHUNK = 250
P_DEGRADE = 0.75
LEVEL_SCALE = 8.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(CPUS)


def to_image(x: np.ndarray, gaps: np.ndarray, levels: bool = False) -> np.ndarray:
    """uint8 cadences (n, 6, 16, 128) -> float16 images (n, 2 or 3, 128, 128).

    With levels, channel 2 holds each panel's overall brightness (log2 power relative to the
    cadence mean, x8), which the row normalization removes but a scintillating star raises.
    """
    z, valid = normalize(x)
    n = len(z)
    img = np.zeros((n, 3 if levels else 2, H, N_CHAN), np.float16)
    lvl = panel_levels(x)[[f"lvl_p{p}" for p in range(6)]].to_numpy() * LEVEL_SCALE if levels else None
    sign = np.where(ON, 1.0, -1.0)
    for i in range(n):
        for p in range(6):
            r0 = p * (N_ROWS + gaps[i])
            img[i, 0, r0:r0 + N_ROWS] = np.clip(z[i, p], -6, 30)
            img[i, 1, r0:r0 + N_ROWS] = sign[p] * valid[i, p][:, None]
            if levels:
                img[i, 2, r0:r0 + N_ROWS] = lvl[i, p] * valid[i, p][:, None]
    return img


def _job(args: tuple) -> np.ndarray:
    name, start, stop, mode, seed, levels = args
    sp = load_split(name)
    x = np.asarray(sp.x[start:stop])
    if mode == "fixed":
        x = degrade(x, np.random.default_rng([SEED, start]), gain_jitter=levels)
    elif mode == "random":
        rng = np.random.default_rng([seed, start])
        d = degrade(x, rng, gain_jitter=levels)
        pick = rng.random(len(x)) < P_DEGRADE
        x = np.where(pick[:, None, None, None], d, x)
    return to_image(x, sp.gaps[start:stop], levels)


def images(pool: ProcessPoolExecutor, name: str, mode: str, seed: int = 0, levels: bool = False) -> np.ndarray:
    n = len(load_split(name).meta)
    jobs = [(name, a, min(a + CHUNK, n), mode, seed, levels) for a in range(0, n, CHUNK)]
    return np.concatenate(list(pool.map(_job, jobs)))


def block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )


class Net(nn.Module):
    def __init__(self, in_ch: int = 2) -> None:
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.b1, self.b2, self.b3, self.b4 = block(32, 48), block(48, 96), block(96, 160), block(160, 256)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, len(LABELS)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.max_pool2d(self.stem(x), 2)          # 64
        x = F.max_pool2d(self.b1(x), 2)            # 32
        x = F.max_pool2d(self.b2(x), 2)            # 16
        x = F.max_pool2d(self.b3(x), 2)            # 8
        x = self.b4(x)
        x = torch.cat([x.mean((2, 3)), x.amax((2, 3))], 1)
        return self.head(x)


class Pretrained(nn.Module):
    """A timm backbone with public ImageNet weights (allowed by rule 02); the 128 x 128 image is
    resized to size x size on the GPU. timm adapts the pretrained stem to 2 or 3 input channels."""

    def __init__(self, arch: str, in_ch: int, size: int) -> None:
        super().__init__()
        import timm
        self.size = size
        self.net = timm.create_model(arch, pretrained=True, in_chans=in_ch, num_classes=len(LABELS), drop_rate=0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.size:
            x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        return self.net(x)


def predict(model: nn.Module, img: np.ndarray, tta: bool = True, amp: bool = False) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast(DEVICE.type, enabled=amp):
        for a in range(0, len(img), 256):
            xb = torch.from_numpy(img[a:a + 256].astype(np.float32)).to(DEVICE)
            p = F.softmax(model(xb).float(), 1)
            if tta:
                p = 0.5 * (p + F.softmax(model(torch.flip(xb, [3])).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds", type=int, nargs="+", default=[0, 1, 2, 3, 4])
    ap.add_argument("--epochs", type=int, default=14)
    ap.add_argument("--name", default="cnn")
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--levels", action="store_true", help="add the panel brightness channel and PKS gain jitter")
    ap.add_argument("--arch", default="small", help="'small' (Net above) or a timm model, e.g. efficientnet_b0")
    ap.add_argument("--size", type=int, default=256, help="input size for a timm model")
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--limit-batches", type=int, default=None, help="smoke test: batches per epoch")
    args = ap.parse_args()
    amp = args.arch != "small" and DEVICE.type == "cuda"

    tr, te = load_split("train"), load_split("test")
    y, folds = tr.y, load_folds()
    w = torch.tensor(len(y) / (len(LABELS) * np.bincount(y, minlength=len(LABELS))), dtype=torch.float32).to(DEVICE)
    pool = ProcessPoolExecutor(CPUS)
    print("device:", DEVICE, " cpus:", CPUS, flush=True)
    (ROOT / "oof").mkdir(exist_ok=True)
    val_img = images(pool, "train", "fixed", levels=args.levels)
    test_img = images(pool, "test", "none", levels=args.levels)
    oof = np.full((len(y), len(LABELS)), np.nan)
    test = np.zeros((len(test_img), len(LABELS)))

    for k in args.folds:
        torch.manual_seed(args.seed + k)
        rng = np.random.default_rng(args.seed + k)
        fit = np.where(folds != k)[0]
        va = np.where(folds == k)[0]
        in_ch = 3 if args.levels else 2
        model = (Net(in_ch) if args.arch == "small" else Pretrained(args.arch, in_ch, args.size)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
        steps = args.epochs * (args.limit_batches or int(np.ceil(len(fit) / 128)))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=steps, pct_start=0.15)
        scaler = torch.cuda.amp.GradScaler(enabled=amp)
        for ep in range(args.epochs):
            t = time.time()
            ep_img = images(pool, "train", "random", seed=args.seed * 1000 + k * 100 + ep, levels=args.levels)
            model.train()
            order = rng.permutation(fit)
            n_b = args.limit_batches or int(np.ceil(len(order) / 128))
            loss_sum = 0.0
            for b in range(n_b):
                idx = order[b * 128:(b + 1) * 128]
                xb = torch.from_numpy(ep_img[idx].astype(np.float32))
                flip = torch.from_numpy(rng.random(len(idx)) < 0.5)
                xb[flip] = torch.flip(xb[flip], [3])
                xb, yb = xb.to(DEVICE), torch.from_numpy(y[idx]).to(DEVICE)
                with torch.autocast(DEVICE.type, enabled=amp):
                    loss = F.cross_entropy(model(xb).float(), yb, weight=w, label_smoothing=0.05)
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                loss_sum += loss.item()
            p = predict(model, val_img[va], tta=False, amp=amp)
            print(f"fold {k} ep {ep}: loss {loss_sum / n_b:.4f}  val F1 "
                  f"{f1_score(y[va], p.argmax(1), average='macro'):.4f}  {time.time() - t:.0f}s", flush=True)
        oof[va] = predict(model, val_img[va], amp=amp)
        test += predict(model, test_img, amp=amp) / len(args.folds)
        print(f"fold {k} TTA val F1 {f1_score(y[va], oof[va].argmax(1), average='macro'):.4f}", flush=True)
        torch.save(model.state_dict(), ROOT / f"oof/{args.name}_f{k}.pt")

    done = ~np.isnan(oof[:, 0])
    print(f"OOF macro-F1 (degraded, {done.sum()} rows): {f1_score(y[done], oof[done].argmax(1), average='macro'):.4f}")
    cols = pd.DataFrame(oof, columns=LABELS)
    pd.concat([tr.meta[["id"]], cols], axis=1).to_csv(ROOT / f"oof/{args.name}_oof.csv", index=False)
    pd.concat([te.meta[["id"]], pd.DataFrame(test, columns=LABELS)], axis=1).to_csv(
        ROOT / f"oof/{args.name}_test.csv", index=False)
    pool.shutdown()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/make_submission.py
"""Blend model probabilities and write submissions/<out>.csv.

    python scripts/make_submission.py --models lgbm --out 01_lgbm
    python scripts/make_submission.py --models lgbm cnn --weights 0.5 0.5 --out 02_blend
    python scripts/make_submission.py --models lgbm cnn --em 0.5 --out 03_blend_em
    python scripts/make_submission.py --models stack4 --bias --out 10_stack4_bias

Blend: weighted mean of log-probabilities, renormalized. Every model is trained with balanced
class weights, so its implied class prior is uniform.

--em ALPHA estimates the test class mix from the unlabeled test predictions with the
Saerens-Latinne-Decaestecker EM (no labels involved), then multiplies the posteriors by
(test prior / uniform) ** ALPHA before the argmax. ALPHA 0 is off, 1 is the full Bayes shift.

--bias adds one log-probability offset per class, chosen by coordinate ascent to maximize OOF
macro-F1 (macro-F1 rewards trading a little accuracy on easy classes for recall on hard ones).
The gain is first measured honestly: offsets tuned on 4 folds, scored on the 5th. Test uses
offsets tuned on all OOF rows.

--shift ALPHA is meant for use with --bias: the OOF-tuned offsets carry the train class mix, so
this adds ALPHA * log(EM test prior / train class share) to move them toward the test mix instead.
"""

import argparse
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import DATA, LABELS, ROOT, load_folds, load_split  # noqa: E402

BIAS_GRID = np.linspace(-2, 2, 41)


def blend(models: list[str], weights: list[float], part: str) -> tuple[pd.Series, np.ndarray]:
    logp, ids = 0.0, None
    for m, w in zip(models, weights):
        f = pd.read_csv(ROOT / f"oof/{m}_{part}.csv")
        ids = f["id"]
        logp = logp + w * np.log(np.clip(f[LABELS].to_numpy(), 1e-9, 1))
    p = np.exp(logp - logp.max(1, keepdims=True))
    return ids, p / p.sum(1, keepdims=True)


def em_prior(p: np.ndarray, iters: int = 200) -> np.ndarray:
    k = p.shape[1]
    prior = np.full(k, 1 / k)
    for _ in range(iters):
        q = p * prior * k
        q /= q.sum(1, keepdims=True)
        prior = q.mean(0)
    return prior


def macro_f1(y: np.ndarray, pred: np.ndarray, k: int) -> float:
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    return float(np.mean(2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)))


def tune_bias(logp: np.ndarray, y: np.ndarray, rounds: int = 3) -> np.ndarray:
    k = logp.shape[1]
    b = np.zeros(k)
    best = macro_f1(y, logp.argmax(1), k)
    for _ in range(rounds):
        for c in range(k):
            for v in BIAS_GRID:
                trial = b.copy()
                trial[c] = v
                s = macro_f1(y, (logp + trial).argmax(1), k)
                if s > best + 1e-6:
                    best, b = s, trial
    return b - b.mean()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--weights", nargs="+", type=float, default=None)
    ap.add_argument("--em", type=float, default=0.0)
    ap.add_argument("--bias", action="store_true", help="tune per-class log offsets on OOF macro-F1")
    ap.add_argument("--shift", type=float, default=0.0, help="move offsets from the train mix toward the EM test mix")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    weights = args.weights or [1 / len(args.models)] * len(args.models)

    y = load_split("train").y
    _, oof = blend(args.models, weights, "oof")
    ok = ~np.isnan(oof).any(1)
    print(f"OOF macro-F1 (degraded train, {ok.sum()} rows): "
          f"{f1_score(y[ok], oof[ok].argmax(1), average='macro'):.4f}")

    bias = np.zeros(len(LABELS))
    if args.bias:
        logp, folds = np.log(np.clip(oof, 1e-9, 1)), load_folds()
        pred = np.zeros(len(y), dtype=int)
        for f in range(5):
            fit, val = (folds != f) & ok, (folds == f) & ok
            pred[val] = (logp[val] + tune_bias(logp[fit], y[fit])).argmax(1)
        print(f"OOF macro-F1 with bias (tuned on other folds): {f1_score(y[ok], pred[ok], average='macro'):.4f}")
        bias = tune_bias(logp[ok], y[ok])
        print("bias:", dict(zip(LABELS, bias.round(2))))
        print("per-class F1 with bias:", dict(zip(LABELS, f1_score(y[ok], pred[ok], average=None).round(3))))

    ids, p = blend(args.models, weights, "test")
    prior = em_prior(p)
    print("EM test prior:", dict(zip(LABELS, prior.round(3))))
    if args.em:
        p = p * (prior * len(LABELS)) ** args.em
    if args.shift:
        bias = bias + args.shift * np.log(prior / (np.bincount(y, minlength=len(LABELS)) / len(y)))
        print("offsets after shift:", dict(zip(LABELS, bias.round(2))))
    pred = np.array(LABELS)[(np.log(np.clip(p, 1e-9, 1)) + bias).argmax(1)]

    sub = pd.DataFrame({"id": ids, "label": pred})
    sample = pd.read_csv(DATA / "sample_submission.csv")
    assert len(sub) == len(sample) and set(sub["id"]) == set(sample["id"]) and sub["id"].is_unique
    sub = sample[["id"]].merge(sub, on="id", how="left")
    assert sub["label"].isin(LABELS).all()
    (ROOT / "submissions").mkdir(exist_ok=True)
    sub.to_csv(ROOT / f"submissions/{args.out}.csv", index=False)
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
    print("predicted mix:", sub["label"].value_counts().to_dict())
    print(f"wrote submissions/{args.out}.csv at commit {commit}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/stack.py
"""Second-level LightGBM on the base models' OOF probabilities (plus chosen raw features).

Same folds as the base models: the stacker for fold k trains on the other folds' OOF rows and
predicts fold k, so its own OOF score is honest. Test predictions are the mean over the 5 fold
stackers applied to the base models' test probabilities.
    python scripts/stack.py --models lgbm3 cnn cnn_s7 --features fit_ trk_ --name stack
"""

import argparse
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from sig.data import CPUS, FEATURES, LABELS, ROOT, SEED, load_folds, load_split  # noqa: E402

PARAMS = dict(
    objective="multiclass", num_class=len(LABELS), learning_rate=0.03, num_leaves=15,
    min_data_in_leaf=60, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=5.0, n_jobs=CPUS, verbose=-1, seed=SEED,
)


def inputs(models: list[str], prefixes: list[str], part: str, feat_file: str) -> pd.DataFrame:
    cols = []
    for m in models:
        p = pd.read_csv(ROOT / f"oof/{m}_{part}.csv")[LABELS]
        cols.append(np.log(p.clip(1e-6, 1)).add_prefix(f"{m}__"))
    if prefixes:
        f = pd.read_parquet(FEATURES / feat_file)
        cols.append(f[[c for c in f.columns if c.startswith(tuple(prefixes))]].reset_index(drop=True))
    return pd.concat(cols, axis=1)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--features", nargs="*", default=[], help="raw feature name prefixes to add")
    ap.add_argument("--name", default="stack")
    args = ap.parse_args()

    tr, te = load_split("train"), load_split("test")
    y, folds = tr.y, load_folds()
    x = inputs(args.models, args.features, "oof", "train_deg.parquet")
    x_te = inputs(args.models, args.features, "test", "test.parquet")
    oof = np.zeros((len(y), len(LABELS)))
    test = np.zeros((len(x_te), len(LABELS)))
    for k in range(5):
        fit = folds != k
        d_fit = lgb.Dataset(x[fit], y[fit], weight=compute_sample_weight("balanced", y[fit]))
        d_val = lgb.Dataset(x[~fit], y[~fit], weight=compute_sample_weight("balanced", y[~fit]))
        m = lgb.train(PARAMS, d_fit, 3000, valid_sets=[d_val], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[~fit] = m.predict(x[~fit], num_iteration=m.best_iteration)
        test += m.predict(x_te, num_iteration=m.best_iteration) / 5
        print(f"fold {k}: iters {m.best_iteration}  F1 {f1_score(y[~fit], oof[~fit].argmax(1), average='macro'):.4f}", flush=True)
    print(f"stack OOF macro-F1 (degraded): {f1_score(y, oof.argmax(1), average='macro'):.4f}  inputs {x.shape[1]}")
    print(classification_report(y, oof.argmax(1), target_names=LABELS, digits=3))
    pd.concat([tr.meta[["id"]], pd.DataFrame(oof, columns=LABELS)], axis=1).to_csv(ROOT / f"oof/{args.name}_oof.csv", index=False)
    pd.concat([te.meta[["id"]], pd.DataFrame(test, columns=LABELS)], axis=1).to_csv(ROOT / f"oof/{args.name}_test.csv", index=False)


if __name__ == "__main__":
    main()


## 1. Folds and features (train, PKS-degraded train with gain jitter, test)

In [ ]:
!python scripts/make_folds.py
!python scripts/build_features.py --gain-jitter

## 2. LightGBM on original + degraded train

In [ ]:
!python scripts/train_lgbm.py --name lgbm4

## 3. CNNs on the global-time image (GPU, about 1 hour each)

In [ ]:
!python scripts/train_cnn.py --name cnn

In [ ]:
!python scripts/train_cnn.py --name cnn_s7 --seed 7 --epochs 18

In [ ]:
!python scripts/train_cnn.py --name cnn_lvl --levels --epochs 18

## 4. Stack of the small models (upload 04)

In [ ]:
!python scripts/stack.py --models lgbm4 cnn cnn_s7 cnn_lvl --features fit_ trk_ lvl_ --name stack5
!python scripts/make_submission.py --models stack5 --bias --out 11_stack5_bias
!python scripts/make_submission.py --models stack5 --bias --shift 0.5 --out 14_stack5_bias_shift05

## 5. ImageNet-pretrained EfficientNets (GPU, about 70 minutes for B0, 2 hours for B2)

In [ ]:
!python scripts/train_cnn.py --arch efficientnet_b0 --levels --lr 1e-3 --epochs 12 --name eff_lvl

In [ ]:
!python scripts/stack.py --models lgbm4 cnn cnn_s7 cnn_lvl eff_lvl --features fit_ trk_ lvl_ --name stack6
!python scripts/make_submission.py --models stack6 --bias --out 16_stack6_bias
!python scripts/make_submission.py --models stack6 --bias --shift 0.5 --out 17_stack6_bias_shift05

In [ ]:
!python scripts/train_cnn.py --arch efficientnet_b2 --levels --lr 1e-3 --epochs 14 --name effb2_lvl

In [ ]:
!python scripts/train_cnn.py --arch efficientnet_b0 --levels --lr 1e-3 --epochs 12 --seed 7 --name eff_lvl_s7

## 6. Final stack of everything

In [ ]:
!python scripts/stack.py --models lgbm4 cnn cnn_s7 cnn_lvl eff_lvl effb2_lvl eff_lvl_s7 --features fit_ trk_ lvl_ --name stack7
!python scripts/make_submission.py --models stack7 --bias --out 18_stack7_bias
!python scripts/make_submission.py --models stack7 --bias --shift 0.5 --out 19_stack7_bias_shift05

## 7. Pack everything for download

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip signal-hunters/oof signal-hunters/submissions && ls -la /kaggle/working